# LangCalc training

<a target="_blank" href="https://github.com/jedick/LangCalc"><img src="https://raw.githubusercontent.com/jedick/LangCalc/main/assets/langcalc-icon-outline.svg" alt="LangCalc icon" width="100"></a>

Training notebook for the [LangCalc](https://github.com/jedick/LangCalc) voice calculator function-calling model. This notebook is adapted from [Fine-tuning with FunctionGemma](https://github.com/google-gemma/cookbook/blob/main/docs/functiongemma/finetuning-with-functiongemma.ipynb).

<a target="_blank" href="https://colab.research.google.com/github/jedick/LangCalc/blob/main/model/notebooks/LangCalc-training.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>

<a target="_blank" href="https://github.com/jedick/LangCalc/blob/main/model/notebooks/LangCalc-training.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>


## Setup development environment

The first step is to install PyTorch and Hugging Face libraries, including TRL for model fine-tuning with different RLHF and alignment techniques.

In [ ]:
# Install PyTorch and Hugging Face libraries
%pip install -q torch transformers datasets trl

# COMMENT IN: if you are running on a GPU that supports BF16 data type and flash attn, such as NVIDIA L4 or NVIDIA A100
#%pip install flash-attn

> _Note: If you are using a GPU with Ampere architecture (such as NVIDIA L4) or newer, you can use Flash attention. Flash Attention is a method that significantly speeds computations up and reduces memory usage from quadratic to linear in sequence length, leading to accelerating training up to 3x. Learn more at [FlashAttention](https://github.com/Dao-AILab/flash-attention/tree/main)._

Before you can start training, you have to make sure that you accepted the terms of use for Gemma. You can accept the license on [Hugging Face](http://huggingface.co/google/functiongemma-270m-it) by clicking on the **Agree** and access repository button on the model page at: http://huggingface.co/google/functiongemma-270m-it

After you have accepted the license, you need a valid Hugging Face Token to access the model. If you are running inside a Google Colab, you can securely use your Hugging Face Token using the Colab secrets otherwise you can set the token as directly in the `login` method. Make sure your token has write access too, as you push your model to Hugging Face Hub after fine-tuning.

In [ ]:
# Log in to the Hugging Face Hub, using a Colab secret if available,
# otherwise falling back to an interactive prompt.
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

if token:
    login(token)
    print("Logged in to Hugging Face Hub using the HF_TOKEN Colab secret.")
else:
    login()

The next cell defines the model name for our fine-tuned model. The fine-tuned model name is also used as the checkpoint directory if we want to save our model to a directory.

You can keep the results on Colab's local virtual machine. However, it is highly recommended saving your intermediate results to your Google Drive. This ensures your training results are safe and allows you to easily compare and select the best model.

*Updates by Jeffrey Dick:*
- Disable Google Drive by default because we push the fine-tuned model directly to the Hub.
- Add language dropdown; construct `model_name` from that.
- Set `dataset_language` to one language code (`en`, `zh`) or to several codes joined by
  periods (`en.zh`, `en.es.zh`) to train on the combined datasets. Each code selects the Hugging Face dataset repo `jedick/langcalc-<code>`.


In [ ]:
dataset_language = "en" #@param ["en", "zh", "en.zh"] {allow-input: true}
model_name = f"functiongemma-langcalc-{dataset_language}"
output_dir = model_name

mount_google_drive = False #@param {type:"boolean"}

if mount_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    output_dir = f"/content/drive/MyDrive/{model_name}"

print(f"Checkpoints will be saved to {output_dir}")

## Tool definitions

We first define the tools that the model can use.
This needs to come before the dataset preparation because that step uses the tool schemas to build a dataset in the conversational tool-calling format.

In [ ]:
import json
import math
import re

import pandas as pd
from transformers.utils import get_json_schema

# --- Tool definitions ---

def add(x: float, y: float):
    """
    Adds two numbers together (sum, total, plus).

    Args:
        x: the first number
        y: the second number

    Returns:
        result: the sum of x and y (x + y).
    """
    return {"result": x + y}


def subtract(x: float, y: float):
    """
    Subtracts one number from another (difference, minus).

    Args:
        x: the starting number
        y: the number to subtract from x

    Returns:
        result: the difference between x and y (x - y).
    """
    return {"result": x - y}


def multiply(x: float, y: float):
    """
    Multiplies two numbers together (product, times).

    Args:
        x: the first number
        y: the second number

    Returns:
        result: the product of x and y (x * y).
    """
    return {"result": x * y}


def divide(x: float, y: float):
    """
    Divides one number by another (quotient, over, go into).

    Args:
        x: the numerator
        y: the denominator

    Returns:
        result: the quotient of x divided by y (x / y).
    """
    return {"result": x / y}


TOOLS = [get_json_schema(t) for t in (add, subtract, multiply, divide)]

DEFAULT_SYSTEM_MSG = "You are a model that can do function calling with the following functions"

## Dataset preparation

Here we load the dataset from the HF Hub. If `dataset_language` lists several languages, the
dataset for each language is loaded from its own repo and the datasets are combined.
The source files used to create these datasets are in the [LangCalc GitHub repo](https://github.com/jedick/LangCalc/tree/main/model/data).

In [ ]:
from datasets import DatasetDict, concatenate_datasets, load_dataset

# dataset_language is one language code (e.g. "en") or several joined by periods
# (e.g. "en.es.zh"). Each code has its own Hugging Face dataset repo, jedick/langcalc-<code>.
languages = dataset_language.split(".")

# Adds the system message and tool schema to the raw examples for fine-tuning.
def to_conversation(example, language):
    """Turn one raw (category, prompt, function, x, y) example into the
    conversational tool-calling format used for fine-tuning."""
    return {
        "messages": [
            {"role": "developer", "content": DEFAULT_SYSTEM_MSG},
            {"role": "user", "content": example["prompt"]},
            {
                "role": "assistant",
                "tool_calls": [
                    {
                        "type": "function",
                        "function": {
                            "name": example["function"],
                            "arguments": {"x": float(example["x"]), "y": float(example["y"])},
                        },
                    }
                ],
            },
        ],
        "tools": TOOLS,
        # keep the language so check_success_rate() can report results for each language
        "language": language,
        # keep the ground-truth values around (unpacked from tool_calls)
        # for use in check_success_rate()
        "expected_function": example["function"],
        "expected_x": float(example["x"]),
        "expected_y": float(example["y"]),
    }

# Load the dataset for each language.
# Each language is converted before the datasets are combined so that the column types
# (which can differ between repos, e.g. int vs float) match across languages.
per_language = []
for language in languages:
    raw_dataset = load_dataset(f"jedick/langcalc-{language}")
    # Drop the original columns so a 'prompt' column doesn't stick around, otherwise SFTTrainer
    # treats this as a prompt-completion dataset and goes looking for a completion column
    per_language.append(
        raw_dataset.map(
            to_conversation,
            fn_kwargs={"language": language},
            remove_columns=raw_dataset["train"].column_names,
        )
    )

# Combine the languages, keeping the train and test splits separate. Examples are in the order
# the languages are listed. Training shuffles them every epoch (train_sampling_strategy="random"
# in SFTConfig below), so this order does not affect training.
dataset = DatasetDict(
    {split: concatenate_datasets([d[split] for d in per_language]) for split in ["train", "test"]}
)

print(f"Train examples: {len(dataset['train'])}")
print(f"Test examples: {len(dataset['test'])}")

## Load the model and inspect prompt formatting

The following code loads the FunctionGemma model and tokenizer from Hugging Face. It then prints
the first training example, both as stored in the dataset and after the tokenizer's chat template
has been applied, to show what the model sees during fine-tuning.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model = "google/functiongemma-270m-it"

# Load the base model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(base_model)

print(f"Device: {model.device}")
print(f"DType: {model.dtype}")

# Print the first training example as stored in the dataset. With several languages combined,
# this example comes from the first language listed in dataset_language.
print("--- dataset input ---")
print(json.dumps(dataset["train"][0], indent=2, ensure_ascii=False))
# Apply the chat template to the same example. The result includes the tool schemas, the
# messages, and the assistant's tool call, formatted as they are for fine-tuning.
debug_msg = tokenizer.apply_chat_template(
    dataset["train"][0]["messages"],
    tools=dataset["train"][0]["tools"],
    add_generation_prompt=False,
    tokenize=False,
)
print("--- Formatted prompt ---")
print(debug_msg)

## Evaluate the base model

Before fine-tuning, we check how well the base model calls the right function with the right
arguments on the test set. The next cell defines the evaluation helpers, including
`check_success_rate()`, which we also use after fine-tuning. It prints each failed test example
with its language and reports the number of successes separately for each language.

The output below shows that the out-of-the-box capabilities may not be good enough for this
use case.

In [ ]:
# Unlike the original example (which only checks whether the right tool
# name shows up in the output), our tool arguments (x, y) are numeric
# and have a single deterministic correct value for every prompt. So we
# run two independent checks per example:
#   1. correct function name
#   2. correct arguments (x and y)
# Both must pass for the example to count as correct.
#
# Each test example has a 'language' column (added in to_conversation()), and
# check_success_rate() reports the results separately for each language.

def extract_tool_calls(text):
    """Parses FunctionGemma's <start_function_call> ... <end_function_call>
    output into a list of {"name": ..., "arguments": {...}} dicts."""

    def cast(v):
        try:
            return int(v)
        except ValueError:
            try:
                return float(v)
            except ValueError:
                return {"true": True, "false": False}.get(v.lower(), v.strip("'\""))

    return [
        {
            "name": name,
            "arguments": {
                k: cast((v1 or v2).strip())
                for k, v1, v2 in re.findall(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]*))", args)
            },
        }
        for name, args in re.findall(
            r"<start_function_call>call:(\w+)\{(.*?)\}<end_function_call>", text, re.DOTALL
        )
    ]


def numbers_match(actual, expected, tol=1e-6):
    if actual is None:
        return False
    try:
        return math.isclose(float(actual), float(expected), rel_tol=tol, abs_tol=tol)
    except (TypeError, ValueError):
        return False


def check_success_rate():
    """Generate a tool call for every test example, print the examples that fail, and
    report the number of successes separately for each language."""
    success_counts = dict.fromkeys(languages, 0)
    total_counts = dict.fromkeys(languages, 0)
    for idx, item in enumerate(dataset["test"]):
        language = item["language"]
        total_counts[language] += 1
        messages = [
            item["messages"][0],
            item["messages"][1],
        ]

        inputs = tokenizer.apply_chat_template(
            messages, tools=TOOLS, add_generation_prompt=True, return_dict=True, return_tensors="pt"
        )

        out = model.generate(
            **inputs.to(model.device), pad_token_id=tokenizer.eos_token_id, max_new_tokens=128
        )
        output = tokenizer.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=False)

        expected_function = item["expected_function"]
        expected_x = item["expected_x"]
        expected_y = item["expected_y"]

        calls = extract_tool_calls(output)
        first_call = calls[0] if calls else None

        actual_function = first_call["name"] if first_call else None
        actual_x = first_call["arguments"].get("x") if first_call else None
        actual_y = first_call["arguments"].get("y") if first_call else None

        func_ok = actual_function == expected_function
        args_ok = numbers_match(actual_x, expected_x) and numbers_match(actual_y, expected_y)

        func_symbol = "✅" if func_ok else "❌"
        args_symbol = "✅" if args_ok else "❌"

        if func_ok and args_ok:
            #print("  -> ✅ correct!")
            success_counts[language] += 1
        else:
            print(f"{idx + 1} Prompt ({language}): {item['messages'][1]['content']}")
            #print(f"  Output: {output}")
            print(
                f"  Function: {func_symbol} (expected '{expected_function}', got "
                f"{repr(actual_function)})"
            )
            print(
                f"  Arguments: {args_symbol} (expected x={expected_x}, y={expected_y}, got "
                f"x={actual_x}, y={actual_y})"
            )
            #print(f"  -> {func_symbol}{args_symbol} failed")

    for language in languages:
        print(f"Success ({language}): {success_counts[language]} / {total_counts[language]}")

In [ ]:
check_success_rate()

## Training

You are now ready to fine-tune your model. Hugging Face TRL
[SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) makes it straightforward to supervise
fine-tune open LLMs. The `SFTTrainer` is a subclass of the `Trainer` from the `transformers`
library and supports all the same features.

Before you can start your training, you need to define the hyperparameters you want to use in
a `SFTConfig` instance.

*Update by Jeffrey Dick:*
- Calculate the number of epochs as `24/len(languages)` so any selection of language(s) has the same number of examples seen during training (dataset size x num epochs).

In [ ]:
from trl import SFTConfig

torch_dtype = model.dtype
num_train_epochs = 24/len(languages)

args = SFTConfig(
    output_dir=output_dir,                  # directory to save model checkpoints
    max_length=512,                         # max sequence length for model and packing of the dataset
    packing=False,                          # Groups multiple samples in the dataset into a single sequence
    num_train_epochs=num_train_epochs,      # number of training epochs
    per_device_train_batch_size=8,          # batch size per device during training
    train_sampling_strategy="random",       # shuffle each epoch (this is the default)
    gradient_checkpointing=False,           # Caching is incompatible with gradient checkpointing
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_steps=1,                        # log every step
    save_strategy="no",                     # don't save checkpoint every epoch
    eval_strategy="epoch",                  # evaluate checkpoint every epoch
    learning_rate=5e-5,                     # learning rate
    fp16=True if torch_dtype == torch.float16 else False,   # use float16 precision
    bf16=True if torch_dtype == torch.bfloat16 else False,  # use bfloat16 precision
    lr_scheduler_type="constant",            # use constant learning rate scheduler
)

You now have every building block you need to create your `SFTTrainer` to start the training of your model.

In [ ]:
from trl import SFTTrainer

# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer,
)

Start training by calling the `train()` method.

In [ ]:
# Start training
trainer.train()

To plot the training and validation losses, you would typically extract these values from the `TrainerState` object or the logs generated during training.

Libraries like Matplotlib can then be used to visualize these values over training steps or epochs. The x-axis would represent the training steps or epochs, and the y-axis would represent the corresponding loss values.

In [ ]:
import matplotlib.pyplot as plt

# Access the log history
log_history = trainer.state.log_history

# Extract training / validation loss
train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

# Plot the training loss
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

## Test Model Inference

After the training is done, you'll want to evaluate and test your model. You can load different samples from the test dataset and evaluate the model on those samples.


In [ ]:
check_success_rate()

## Optional: Upload the model to Hugging Face Hub

If you're satisfied with the model performance, upload it to Hugging Face Hub so you easily share your model or access it later -- for example, in the [LangCalc inference notebook](https://github.com/jedick/LangCalc/blob/main/model/notebooks/LangCalc-inference.ipynb).

In [ ]:
from huggingface_hub import ModelCard, ModelCardData, whoami

# Save the final model to output_dir
if mount_google_drive:
    trainer.save_model()
    model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")
    tokenizer = AutoTokenizer.from_pretrained(output_dir)

push_to_hub = False #@param {type:"boolean"}

if push_to_hub:
    # Construct HF repo_id
    username = whoami()['name']
    hf_repo_id = f"{username}/{model_name}"

    # Push the model and tokenizer
    repo_url = model.push_to_hub(hf_repo_id, commit_message="Upload model")
    print("Pushed model")
    tokenizer.push_to_hub(hf_repo_id)
    print("Pushed tokenizer")

    # Push a model card
    card_data = ModelCardData(
        base_model=base_model,
        tags=["calculator", "function-calling", "gemma"],
        datasets=[f"jedick/langcalc-{language}" for language in languages],
        language=languages,
        license="gemma",
    )
    icon_html = '<img src="https://raw.githubusercontent.com/jedick/LangCalc/main/assets/langcalc-icon-outline.svg" alt="LangCalc icon" width="100"/>'
    info_text = "More info: https://github.com/jedick/langcalc"
    card_body = f"{icon_html}\n\nA fine-tuned model based on `{base_model}`.\n\n{info_text}"
    card = ModelCard(f"---\n{card_data.to_yaml()}\n---\n\n{card_body}")
    card.push_to_hub(hf_repo_id)
    print("Pushed model card")

    print(f"Committed {repo_url}")